# scikit-learn Baseline: Mass Regression

This notebook builds the first reproducible regression baseline using the exoplanet.eu catalog.

In [2]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

DATA_DIR = Path.cwd() / "data"
eu_df = pd.read_csv(
    DATA_DIR / "exoplanet.eu_catalog_28-05-26_22_04_09.csv", low_memory=False
)
eu_df.head()

,name,planet_status,mass,mass_error_min,mass_error_max,mass_sini,mass_sini_error_min,mass_sini_error_max,radius,radius_error_min,...,star_sp_type,star_age,star_age_error_min,star_age_error_max,star_teff,star_teff_error_min,star_teff_error_max,star_detected_disc,star_magnetic_field,star_alternate_names
0,109 Psc b,Confirmed,6.800,0.300,0.900,6.3830,0.078,0.078,1.152,NaN,...,G5 IV,6.90,0.60,0.60,5600.0,80.000,80.000,NaN,NaN,HD 10697
1,112 Psc b,Confirmed,NaN,0.005,0.004,0.0330,0.005,0.004,NaN,NaN,...,G0IV,NaN,NaN,NaN,5986.0,105.437,105.437,NaN,NaN,HD 12235
2,112 Psc c,Confirmed,9.866,1.781,3.190,NaN,NaN,NaN,NaN,NaN,...,G0IV,NaN,NaN,NaN,5986.0,105.437,105.437,NaN,NaN,HD 12235
3,11 UMi b,Confirmed,NaN,1.100,1.100,11.0873,1.100,1.100,NaN,NaN,...,K4III,1.56,0.54,0.54,4340.0,70.000,70.000,NaN,NaN,NaN
4,14 And Ab,Confirmed,NaN,0.230,0.230,4.6840,0.230,0.230,NaN,NaN,...,K0III,NaN,NaN,NaN,4813.0,20.000,20.000,NaN,NaN,NaN


In [3]:
target = "mass"
numeric_features = [
    "mass_sini",
    "radius",
    "orbital_period",
    "semi_major_axis",
    "eccentricity",
    "inclination",
    "temp_calculated",
    "star_distance",
    "star_metallicity",
    "star_mass",
    "star_radius",
    "star_age",
    "star_teff",
]
categorical_features = [
    "planet_status",
    "detection_type",
    "mass_measurement_type",
    "star_sp_type",
]

model_df = eu_df[[target, *numeric_features, *categorical_features]].copy()
model_df = model_df[model_df[target].notna()]

X = model_df.drop(columns=[target])
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

len(X_train), len(X_test)

(1876, 469)

In [4]:
numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

ridge_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]
)

forest_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ]
)

In [6]:
def regression_report(name, fitted_model, X_test, y_test):
    predictions = fitted_model.predict(X_test)
    rmse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    return {"model": name, "rmse": rmse, "mae": mae, "r2": r2}


ridge_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

pd.DataFrame(
    [
        regression_report("ridge", ridge_model, X_test, y_test),
        regression_report("random_forest", forest_model, X_test, y_test),
    ]
).sort_values("rmse")

,model,rmse,mae,r2
1,random_forest,2.399150,0.826347,0.559992
0,ridge,3.254043,1.174118,0.403203
